In [ ]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import contextily as cx
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Gevraagd wordt een dataframe voor de UNPAVED zoals dit:
#	    id	    total_area	lu_areas	surface_level	soiltype	surface_storage	infiltration_capacity	initial_gwd	meteo_area	px	py	boundary_node
# code												
# 15.0	15.0	1375	250 0 0 0 0 0 0 0 0 0 225 0 0 0 0 0	                16.93   107	10.000	100.000	1.20	15.0	199378	395163	lat_15.0
# 55.0	55.0	303875	124200 18000 0 0 0 0 0 0 0 150 68125 0 11875 0...	21.69	105	10.000	100.000	1.20	55.0	197488	392239	lat_55.0
# 56.0	56.0	13300	5400 0 0 0 0 0 0 0 0 0 4425 0 375 0 1150 0	        20.63	113	10.000	100.000	1.20	56.0	197789	392200	lat_56.0
# 57.0	57.0	60925	6550 725 0 22800 0 0 0 0 0 0 9375 0 4600 0 175 0	21.49	113	10.000	100.000	1.20	57.0	197982	392247	lat_57.0

# en een dataframe voor ernst zoals deze:
# 	    id	    cvo	            lv	        cvi	    cvs
# code					
# 15.0	15.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 55.0	55.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 56.0	56.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 57.0	57.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00

In [ ]:
# For land use and soil type a coding is prescribed. For landuse, the legend of the map is expected to be as follows: <br>
landuse_mapping = {
    "potatoes": 1,
    "wheat": 2,
    "sugar beet": 3,
    "corn": 4,
    "other crops": 5,
    "bulbous plants": 6,
    "orchard": 7,
    "grass": 8,
    "deciduous forest": 9,
    "coniferous forest": 10,
    "nature": 11,
    "barren": 12,
    "open water": 13,
    "built-up": 14,
    "greenhouses": 15
}

# For classes 1-12, the areas are calculated from the provided raster and remapped to the classification in the Sobek RR-tables.


# The coding for the soil types:<br>
soiltype_first_mapping = {
    "podzol": "Podzol (grof zand)"
}

soiltype_mapping = {
    "Veengrond met veraarde bovengrond": 1,
    "Veengrond met veraarde bovengrond, zand": 2,
    "Veengrond met kleidek": 3,
    "Veengrond met kleidek op zand": 4,
    "Veengrond met zanddek op zand": 5,
    "Veengrond op ongerijpte klei": 6,
    "Stuifzand": 7,
    "Podzol (Leemarm, fijn zand)": 8,
    "Podzol (zwak lemig, fijn zand)": 9,
    "Podzol (zwak lemig, fijn zand op grof zand)": 10,
    "Podzol (lemig keileem)": 11,
    "Enkeerd (zwak lemig, fijn zand)": 12,
    "Beekeerd (lemig fijn zand)": 13,
    "Podzol (grof zand)": 14,
    "Zavel": 15,
    "Lichte klei": 16,
    "Zware klei": 17,
    "Klei op veen": 18,
    "Klei op zand": 19,
    "Klei op grof zand": 20,
    "Leem": 21
}

# And surface elevation needs to be in m+NAP.

In [ ]:
def generate_unpaved_df_from_rr_input(gdf):
    df_unpaved = pd.DataFrame()
    df_unpaved["code"] = "unpaved_" + gdf["GFEIDENT"]
    df_unpaved["id"] = "unpaved_" + gdf["GFEIDENT"]
    df_unpaved["total_area"] = gdf["Area_RR_unpaved_m2"].astype(int)
    df_unpaved["lu_areas"] = gdf["Area_RR_unpaved_m2"].astype(int).astype(str) + " 0"*15
    df_unpaved["surface_level"] = gdf["SurfaceLevel_mNAP"]
    df_unpaved["soiltype"] = gdf["CapSimSoilType"].map(soiltype_first_mapping).map(soiltype_mapping) + 100              # soiltype_mapping + 100
    df_unpaved["surface_storage"] = gdf["StorageOnLand_mm"]
    df_unpaved["infiltration_capacity"] = gdf["InfiltrationCapacity_mmph"]
    df_unpaved["initial_gwd"] = gdf["InitialGroundwaterLevel_mBelowSurface"]
    df_unpaved["meteo_area"] = gdf["MeteoStationName"]
    df_unpaved["px"] = gdf.geometry.x
    df_unpaved["py"] = gdf.geometry.y
    df_unpaved["boundary_node"] = "lateral_" + gdf["GFEIDENT"].astype(str)
    df_unpaved = df_unpaved.set_index("code")
    return df_unpaved


def generate_ernst_df_from_rr_input(gdf):
    df_ernst = pd.DataFrame()
    df_ernst["code"] = "ernst_" + gdf["GFEIDENT"].astype(str)
    df_ernst["id"] = "ernst_" + gdf["GFEIDENT"].astype(str)
    df_ernst["cvo"] = gdf.apply(lambda row: " ".join([str(row["FirstDrainResistance_d"]), str(row["SecondDrainResistance_d"]), str(row["ThirdDrainResistance_d"])]), axis=1)
    df_ernst["lv"] = gdf.apply(lambda row: " ".join([str(row["FirstDrainLevel_mNAP"]), str(row["SeconDrainLevel_mNAP"]), str(row["ThirdDrainLevel_mNAP"])]), axis=1)
    df_ernst["cvi"] = gdf["OpenWaterHorizontalInflowResistance_d"].astype(str)
    df_ernst["cvs"] = gdf["SurfaceOverlandFlowResistance_d"].astype(str)
    df_ernst = df_ernst.set_index("code")
    return df_ernst

In [ ]:
# INPUT vanuit WRIJ voor RR unpaved methode
dir_data_input = Path("..\\..\\WRIJ_RR_Unpaved_methode_01_data\\stap2RRinput\\20260424")

seasons = ["zomer", "winter"]
gpkg_rr_input_zomer_winter = ["RR_input_ZOMER.gpkg", "RR_input_WINTER.gpkg"]

dir_first_output = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_tussenresultaat\\RR_input_gdfs\\")

In [ ]:
for season, gpkg_rr_input in zip(seasons, gpkg_rr_input_zomer_winter):
    gdf = gpd.read_file(dir_data_input / gpkg_rr_input)

    df_unpaved = generate_unpaved_df_from_rr_input(gdf)
    df_unpaved.to_csv(dir_first_output / f"gdf_unpaved_{season}.csv")
    display(df_unpaved.head(3))

    df_ernst = generate_ernst_df_from_rr_input(gdf)
    df_ernst.to_csv(dir_first_output / f"gdf_ernst_{season}.csv")
    display(df_ernst.head(3))

In [ ]:
gdf.head()